<a href="https://colab.research.google.com/github/tmacpherson6/investment_advisor_dashboard_capstone/blob/master/Kristine_SCF_Data_Prep_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Survey of Consumer Finances (SCF) Public Data Preparation**

This notebook will prepare data from the **2022 Survey of Consumer Finances** (SCF) available here: https://www.federalreserve.gov/econres/scfindex.html. The SCF is a triennial cross-sectional survey of U.S. families.

In this notebook, we will use the **convey package** in R to prepare the data for analysis. We will utilize code from the following link as reference for downloading and converting the data: https://www.convey-r.org/1.6-survey-of-consumer-finances-scf.html#survey-of-consumer-finances-scf




### **Packages**

In [2]:
# Comment out if libraries below have already been installed
install.packages("haven")
install.packages("stringr")
install.packages("survey")
install.packages("mitools")
install.packages("convey")
install.packages("scf")
install.packages("httr")
install.packages("dplyr")

library(haven)
library(stringr)
library(survey)
library(mitools)
library(convey)
library(scf)
library(httr)
library(dplyr)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘minqa’, ‘numDeriv’, ‘mitools’, ‘RcppArmadillo’


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: grid

Loading required package: Matrix

Loading required package: survival


Attaching package: ‘survey’


The following object is masked from ‘package:graphics’:

    dotchart



Attaching package: ‘scf’


The following object

### **Download the data**
Define a function to download and import each STATA file:

In [3]:
scf_dta_import <-
  function(this_url) {
    this_tf <- tempfile()
    download.file(this_url , this_tf , mode = 'wb')
    this_tbl <- read_dta(this_tf)
    this_df <- data.frame(this_tbl)
    file.remove(this_tf)
    names(this_df) <- tolower(names(this_df))
    this_df
  }

Download and import the full, summary extract, and replicate weights tables

In [4]:
scf_df <-
  scf_dta_import("https://www.federalreserve.gov/econres/files/scf2022s.zip")

ext_df <-
  scf_dta_import("https://www.federalreserve.gov/econres/files/scfp2022s.zip")

scf_rw_df <-
  scf_dta_import("https://www.federalreserve.gov/econres/files/scf2022rw1s.zip")

vars <-

Confirm both the full public data and the summary extract contain five records per family:

In [5]:
stopifnot(nrow(scf_df) == nrow(scf_rw_df) * 5)
stopifnot(nrow(scf_df) == nrow(ext_df))

Confirm only the primary economic unit and the five implicate identifiers overlap:

In [6]:
stopifnot(all(sort(intersect(
  names(scf_df) , names(ext_df)
)) == c('y1' , 'yy1')))

stopifnot(all(sort(intersect(
  names(scf_df) , names(scf_rw_df)
)) == c('y1' , 'yy1')))

stopifnot(all(sort(intersect(
  names(ext_df) , names(scf_rw_df)
)) == c('y1' , 'yy1')))

### **Calculate the replicate weights**

Remove the implicate identifier from the replicate weights table, add a column of fives for weighting:

In [7]:
scf_rw_df[, 'y1'] <- NULL

scf_df[, 'five'] <- 5

Break the main table into five different implicates based on the final character of the column y1:

In [9]:
s1_df <- scf_df[str_sub(scf_df[, 'y1'] ,-1 ,-1) == 1 ,]
s2_df <- scf_df[str_sub(scf_df[, 'y1'] ,-1 ,-1) == 2 ,]
s3_df <- scf_df[str_sub(scf_df[, 'y1'] ,-1 ,-1) == 3 ,]
s4_df <- scf_df[str_sub(scf_df[, 'y1'] ,-1 ,-1) == 4 ,]
s5_df <- scf_df[str_sub(scf_df[, 'y1'] ,-1 ,-1) == 5 ,]

Combine these into a single list, then merge each implicate with the summary extract:

In [10]:
scf_imp <- list(s1_df , s2_df , s3_df , s4_df , s5_df)

scf_list <- lapply(scf_imp , merge , ext_df)

Replace all missing values in the replicate weights table with zeroes, multiply the replicate weights by the multiplication factor, then only keep the unique identifier and the final (combined) replicate weights:

In [11]:
scf_rw_df[is.na(scf_rw_df)] <- 0

scf_rw_df[, paste0('wgt' , 1:999)] <-
  scf_rw_df[, paste0('wt1b' , 1:999)] * scf_rw_df[, paste0('mm' , 1:999)]

scf_rw_df <- scf_rw_df[, c('yy1' , paste0('wgt' , 1:999))]

Sort both the five implicates and also the replicate weights table by the unique identifier:

In [12]:
scf_list <-
  lapply(scf_list , function(w)
    w[order(w[, 'yy1']) ,])

scf_rw_df <- scf_rw_df[order(scf_rw_df[, 'yy1']) ,]

### **Filter out shadow variables**

Each of the variables in the main data set has a "shadow" variable that describes the original state of the variable (i.e., whether it was missing for some reason, a range response was given, etc.). The shadow variables have the same numbers as the main variable, but have a prefix of "j". We will remove the variables as they do not impact the dependent variable.

In [14]:
scf_list <- bind_rows(scf_list) %>% select(!starts_with("j"))

The dependent variable we want to predict is x7557. It represents a survey respondent's willingness to take financial risks on a scale from 0 - 10, with 0 being the lowest willingness to take financial risks. When we check the unique values in x7557, 0 is represented by -1.

In [16]:
#unique(scf_list$x7557)

scf_list$x7557 <- ifelse(scf_list$x7557 == -1, 0, scf_list$x7557)

Save cleaned data as a csv. If you are using Google Colab, you can download the files by clicking the file icon on the left-hand side of the screen.

"scf_list.csv" contains the full dataset including all implicates. Implicates can be filtered out based on the final character in column y1.

"scf_rw_df" contains the replicate weights for each record. The file contains the case ID in yy1, along with 999 replicate-weight columns, one for each of the 999 bootstrap replicate samples used to estimate sampling variability.

More information about the replicate weights and imputations are available here: https://www.federalreserve.gov/econres/files/Standard_Error_Documentation.pdf

In [17]:
# SCF Variables
write.csv(scf_list, "scf_list.csv", row.names = FALSE)

# Replicate weights table
write.csv(scf_rw_df, "scf_rw_df.csv", row.names = FALSE)